# **Setup**

In [ ]:
# Estou utilizando o drive para carregar os datasets direto de lá
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install torch
!pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.6.0+cu124.html
!pip install torch-geometric
!pip install torch-geometric-temporal
!pip install numpy pandas tqdm scikit-learn matplotlib loguru torchmetrics timesfm

In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/ColabData/LABIA/DatasetsTSFM')

In [ ]:
# basic
import os
import pickle
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
# pre processing
from sklearn import preprocessing as pre
# NN
import torch
import torch.nn as nn
from torch import Tensor
import torch.nn.functional as F
import torch.optim as optim
from torch.nn import MSELoss
from torch_geometric.nn import GCNConv
# val and plot
from torchmetrics.regression import R2Score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error
from loguru import logger as log
from val import calculate_metrics
# plot
import matplotlib.pyplot as plt
# foundation model
import timesfm
from functools import reduce



# **Experimento**

In [ ]:
SEED = 1345
def seed_everything(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(SEED)
warnings.filterwarnings('ignore')

In [ ]:
print(torch.__version__)
print(torch.version.cuda)

In [ ]:
def load_datasets(filepath):
    """Carrega os datasets de arquivos pickle."""
    try:
        with open(filepath, 'rb') as f:
            dataset = pickle.load(f)
        return dataset
    except IOError as e:
        log.error(f"Erro ao carregar o dataset: {e}")
    except pickle.PickleError as e:
        log.error(f"Erro ao desserializar o dataset: {e}")
        traceback.print_exception(e)

In [ ]:
test_dataset = load_datasets(f'/content/drive/MyDrive/ColabData/LABIA/DatasetsTSFM/test2.pkl')

In [ ]:
# x (dados de input)
test_dataset[0].x.shape

In [ ]:
inference_size = test_dataset[0].y.shape[1]
inference_size

In [ ]:
# 44783654 ufba ondina portaria 01
# 43768720 estacao da lapa
# 230565994 farol de itapua
# 125960550 estadio barradão
# 45833547 Ferry
# 44784438 fonte nova
# 47568123 shopping barra
# 44072192 tatro castro alves
# 258781031 rodovaria
# 44783914 elevador lacerda

In [ ]:
ids = {53:'125960550', 365:'230565994', 382:'258781031', 666:'43768720', 701:'44072192', 1326:'44783654', 1404:'44783914', 1569:'44784438', 1916:'45833547', 2617:'47568123'}
nodes =  [53, 365, 382, 666, 701, 1326, 1404, 1569, 1916, 2617]

In [ ]:
model_name = 'timesfm-2.0-500m-pytorch'

if model_name == 'timesfm-2.0-500m-pytorch':
  tfm = timesfm.TimesFm(
      hparams=timesfm.TimesFmHparams(
          backend="gpu",
          per_core_batch_size=40,
          horizon_len=inference_size,
          num_layers=50,
          use_positional_embedding=False,
          context_len=2048,

      ),
      checkpoint=timesfm.TimesFmCheckpoint(
                    huggingface_repo_id=(''.join(('google/', model_name))),
      )
)

elif model_name == 'timesfm-1.0-200m-pytorch':
  tfm = timesfm.TimesFm(
      hparams=timesfm.TimesFmHparams(
          backend="gpu",
          per_core_batch_size=40,
          horizon_len=inference_size,
      ),
  checkpoint=timesfm.TimesFmCheckpoint(
                    huggingface_repo_id=(''.join(('google/', model_name))),
      )
  )

In [ ]:
scores_error = {'node':[], 'mae': [], 'mse': [], 'r2': [], 'mape': []}
targets = {}

for node in nodes:
    temp_scores_error = {'node':[], 'mae': [], 'mse': [], 'r2': [], 'mape': []}
    targets[node] = []
    cost, time = 0, 0
    for time, snapshot in tqdm(enumerate(test_dataset)):
        snapshot.to('cpu')

        input_data = np.array(snapshot.x[node, :])
        forecast, experimental_quantile_forecast = tfm.forecast(
            [input_data],
            freq=[0],
        )
        #
        y_hat =  torch.tensor(forecast[0])
        #

        # nao alterar mais abaixo

        cost = cost + torch.mean((y_hat-snapshot.y)**2)
        y_true = snapshot.y.cpu().data.numpy()
        y_pred = y_hat.cpu().data.numpy()

        temp_scores_error['node'].append(node)
        temp_scores_error['mse'].append(mean_squared_error(y_true[node,:], y_pred))
        temp_scores_error['mae'].append(mean_absolute_error(y_true[node,:], y_pred))
        temp_scores_error['r2'].append(r2_score(y_true[node,:], y_pred))
        temp_scores_error['mape'].append(mean_absolute_percentage_error(y_true[node,:], y_pred))

        #targets.append({'true': y_true,
                        #'pred': y_pred})

        targets[node].append({"input":  snapshot.x[node,:].cpu().numpy(),
                              'true': y_true,
                              'pred': y_pred,
                              'node': ids[node]
                             })

    # Calculate and print the averages of each metric
    for metric, values in temp_scores_error.items():
      average = sum(values) / len(values) if values else 0
      scores_error[metric].append(average)
      print(f"Average {metric.upper()}: {average:.4f}")



    cost = cost / (time+1)
    cost = cost.item()
    log.info(f"node: {node} MSE test: {cost:.4f}")


In [ ]:
with open(f'{model_name}-targets-batch.pkl', 'wb') as f:
    pickle.dump(targets, f)

In [ ]:
df_results = pd.DataFrame(scores_error)
df_results

In [ ]:
df_results["model"] = "timesfm-1.0-200m"

In [ ]:
df_results[['model', 'node', 'mae', 'mse', 'r2', 'mape']]

In [ ]:
df_results.to_csv(''.join((model_name, '-batch.csv')))